<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; EDM&plus; Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">EDM&plus; NB03 &mdash; Portfolios, Holdings &amp; Transactions</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Creates 14 transaction portfolios and loads 221 transactions, with Strategy registered as a sub-holding key to segment positions.</div>
</div>

<sub>EDM&plus; pack sequence: NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; <b>NB03</b> &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07</sub>

# NB03: Portfolios, Holdings & Transactions

**What this does:** Creates 14 portfolios and loads 221 transactions (178 instrument trades + 43 cash movements).

**Business context:** Portfolios hold positions. Transactions create and modify those positions. In LUSID, you create the portfolio first, then upsert transactions into it. LUSID calculates holdings automatically from the transaction history (no need to load holdings separately).

**Key concepts:**
- **Sub Holding Keys (SHKs):** The Strategy property is registered as an SHK, which means cash with different strategies appears as separate line items even if the currency is the same
- **Portfolio creation date:** Must predate all transactions. We use 2020-01-01 because transactions go back to 2023

**Verify after running:** Go to Portfolios, filter scope `edm-training-accounts`. Click TPV_GlobalEquities → Holdings tab.

**Manual exercise after this notebook:** Add portfolio properties (Strategy, AssetClass, PortfolioManager) via the Properties tab in the UI.

In [ ]:
# Run this cell first — installs required packages (takes ~30 seconds, only needed once per session)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'lusid-sdk', 'lusid-workflow-sdk', 'finbourne-sdk-utilities', 'lusid-drive-sdk', 'lumipy'])
print("✅ Packages installed")

In [ ]:
# ============================================================================
# CREDENTIALS: edit secrets.json (NOT this notebook)
# ============================================================================
# Copy secrets.json.example to secrets.json and fill in your domain + PAT:
#   {
#     "api_url": "https://<YOUR_DOMAIN>.lusid.com/api",
#     "personal_access_token": "<YOUR_PAT>"
#   }
# To get a PAT: LUSID web app → profile icon (top right) → Personal Access Tokens → Create
# (Override the file location with the EDM_SECRETS_PATH environment variable.)

import json as _json, os as _os
_secrets_path = _os.environ.get("EDM_SECRETS_PATH", "secrets.json")
with open(_secrets_path) as _f:
    _secrets = _json.load(_f)
api_url = _secrets["api_url"]
personal_access_token = _secrets["personal_access_token"]

# ============================================================================
# Imports — do not edit below this line
# ============================================================================
import os, json
from datetime import datetime, timedelta, date, timezone
import datetime as dt
import pandas as pd
import pytz

import lusid as lu
import lusid.models as lm

try:
    import lusid_workflow as lwf
    import lusid_workflow.models as wf_models
except ImportError:
    lwf = None
    wf_models = None
    print("⚠️ lusid_workflow not available")

import lumipy as lp

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.4f}".format

# ============================================================================
# Authentication
# ============================================================================
if "<YOUR_DOMAIN>" in api_url or "<YOUR_PAT>" in personal_access_token or not personal_access_token:
    raise ValueError(
        "\n\n⛔ You need to edit the two lines at the top of this cell:\n"
        "   1. Replace <YOUR_DOMAIN> with your LUSID domain name\n"
        "   2. Replace <YOUR_PAT> with your Personal Access Token\n\n"
        "   To get a PAT: LUSID web app → profile icon → Personal Access Tokens → Create")

def get_factory(app='lusid'):
    module = __import__(app)
    # Each service has a different URL path
    url_map = {'lusid': '/api', 'lusid_workflow': '/workflow', 'lusid_drive': '/drive'}
    service_url = api_url.replace('/api', url_map.get(app, '/api'))
    config_loaders = [module.extensions.ArgsConfigurationLoader(
        api_url=service_url, access_token=personal_access_token)]
    return module.extensions.SyncApiClientFactory(config_loaders=config_loaders)

def get_lumi():
    luminesce_url = api_url.replace('/api', '/honeycomb')
    return lp.get_client(access_token=personal_access_token, api_url=luminesce_url)

# Initialise
lusid_factory = get_factory('lusid')

# Verify connection
try:
    api_instance = lusid_factory.build(lu.ApplicationMetadataApi)
    api_response = api_instance.get_lusid_versions()
    domain = api_response.links[0].href
    env_url = domain[0:domain.find('/app/')]
    print(f"✅ Connected to {env_url}")
    print(f"   API Version: {api_response.build_version}")
except Exception as e:
    print(f"⚠️ Connected but could not verify: {e}")

lumi = get_lumi()
print("✅ Luminesce ready")

---
## Build APIs and Create Portfolios

In [ ]:
portfolios_api = lusid_factory.build(lu.PortfoliosApi)
txn_portfolios_api = lusid_factory.build(lu.TransactionPortfoliosApi)

SCOPE = 'edm-training3'
PORTFOLIO_SCOPE = f'{SCOPE}-accounts'
EFFECTIVE_DATE = '2020-01-01T00:00:00Z'
DATA_DIR = 'data/'
SHK_KEY = f'Transaction/{SCOPE}/Strategy'

# Benchmarks
with open(f'{DATA_DIR}benchmark-portfolios.json') as f:
    benchmarks = json.load(f)

print("--- Creating Benchmark Portfolios ---")
for port_code, config in benchmarks.items():
    try:
        portfolios_api.delete_portfolio(scope=PORTFOLIO_SCOPE, code=port_code)
    except: pass
    try:
        txn_portfolios_api.create_portfolio(
            scope=PORTFOLIO_SCOPE,
            create_transaction_portfolio_request=lm.CreateTransactionPortfolioRequest(
                display_name=config['display_name'], code=port_code,
                base_currency=config['base_currency'],
                description=config.get('description', ''),
                created=EFFECTIVE_DATE, sub_holding_keys=[]))
        print(f"  ✅ Created: {port_code}")
    except lu.ApiException as e:
        if 'PortfolioWithIdAlreadyExists' in str(e.body):
            print(f"  ℹ️  Already exists: {port_code}")
        else:
            print(f"  ❌ Error: {e.reason}")

# Transaction Portfolios
print("\n--- Creating Transaction Portfolios ---")
df_txn = pd.read_csv(f'{DATA_DIR}total-portfolio-view.csv')
df_cash = pd.read_csv(f'{DATA_DIR}cash_transactions.csv')
all_codes = set(df_txn['PortfolioCode'].unique()) | set(df_cash['PortfolioCode'].unique())
print(f"  Found {len(all_codes)} portfolio codes")

for port_code in sorted(all_codes):
    port_txns = df_txn[df_txn['PortfolioCode'] == port_code]
    base_ccy = str(port_txns['PortfolioBaseCurrency'].iloc[0]) if len(port_txns) > 0 else 'USD'
    try:
        portfolios_api.delete_portfolio(scope=PORTFOLIO_SCOPE, code=port_code)
    except: pass
    try:
        txn_portfolios_api.create_portfolio(
            scope=PORTFOLIO_SCOPE,
            create_transaction_portfolio_request=lm.CreateTransactionPortfolioRequest(
                display_name=port_code.replace('TPV_', '').replace('_', ' '),
                code=port_code, base_currency=base_ccy,
                description=f'Training portfolio: {port_code}',
                created=EFFECTIVE_DATE, sub_holding_keys=[SHK_KEY]))
        print(f"  ✅ Created: {port_code}")
    except lu.ApiException as e:
        if 'PortfolioWithIdAlreadyExists' in str(e.body):
            print(f"  ℹ️  Already exists: {port_code}")
        else:
            print(f"  ❌ Error: {e.reason}")

---
## Load Transactions

In [ ]:
print("--- Loading Instrument Transactions ---")
total_loaded = 0
for port_code in sorted(all_codes):
    port_txns = df_txn[df_txn['PortfolioCode'] == port_code]
    if len(port_txns) == 0: continue
    transactions = []
    for idx, row in port_txns.iterrows():
        try: txn_date = pd.to_datetime(str(row['TransactionDate']), dayfirst=True)
        except: txn_date = pd.Timestamp('2025-01-01')
        txn_str = txn_date.strftime('%Y-%m-%dT00:00:00Z')
        settle_str = (txn_date + pd.Timedelta(days=2)).strftime('%Y-%m-%dT00:00:00Z')
        strategy = str(row.get('Strategy', '')).strip()
        props = {}
        if strategy:
            props[f'Transaction/{SCOPE}/Strategy'] = lm.PerpetualProperty(
                key=f'Transaction/{SCOPE}/Strategy', value=lm.PropertyValue(label_value=strategy))
        transactions.append(lm.TransactionRequest(
            transaction_id=str(row.get('TxnId', f'TXN_{idx}')),
            type=str(row.get('Type', 'Buy')).strip(),
            instrument_identifiers={'Instrument/default/ClientInternal': str(row['ClientInternal']).strip()},
            transaction_date=txn_str, settlement_date=settle_str,
            units=float(row.get('Units', 0)),
            transaction_price=lm.TransactionPrice(price=float(row.get('TradePrice', 0)), type='Price'),
            total_consideration=lm.CurrencyAndAmount(
                amount=float(row.get('Units', 0)) * float(row.get('TradePrice', 0)),
                currency=str(row.get('LocalCurrency', 'USD')).strip()),
            transaction_currency=str(row.get('LocalCurrency', 'USD')).strip(),
            properties=props if props else None))
    try:
        txn_portfolios_api.upsert_transactions(scope=PORTFOLIO_SCOPE, code=port_code, transaction_request=transactions)
        total_loaded += len(transactions)
        print(f"  {port_code}: {len(transactions)} transactions")
    except lu.ApiException as e:
        body = e.body[:300] if e.body else e.reason
        print(f"  {port_code}: ERROR {body}")
print(f"  Total: {total_loaded}")

print("\n--- Loading Cash Transactions ---")
cash_loaded = 0
for port_code in df_cash['PortfolioCode'].unique():
    port_cash = df_cash[df_cash['PortfolioCode'] == port_code]
    transactions = []
    for idx, row in port_cash.iterrows():
        try: txn_date = pd.to_datetime(str(row['TransactionDate']), dayfirst=True)
        except: txn_date = pd.Timestamp('2023-12-01')
        txn_str = txn_date.strftime('%Y-%m-%dT00:00:00Z')
        try: settle_str = pd.to_datetime(str(row.get('SettlementDate', row['TransactionDate'])), dayfirst=True).strftime('%Y-%m-%dT00:00:00Z')
        except: settle_str = txn_str
        ccy = str(row['Currency']).strip()
        rate = float(row.get('TradeToPortfolioRate', 1))
        strategy = str(row.get('Strategy', 'Cash')).strip()
        props = {f'Transaction/{SCOPE}/Strategy': lm.PerpetualProperty(
            key=f'Transaction/{SCOPE}/Strategy', value=lm.PropertyValue(label_value=strategy))}
        transactions.append(lm.TransactionRequest(
            transaction_id=str(row.get('TxnId', f'CASH_{idx}')),
            type=str(row.get('Type', 'FundsIn')),
            instrument_identifiers={'Instrument/default/Currency': ccy},
            transaction_date=txn_str, settlement_date=settle_str,
            units=float(row.get('Units', 0)),
            transaction_price=lm.TransactionPrice(price=rate, type='Price'),
            total_consideration=lm.CurrencyAndAmount(amount=float(row.get('Units', 0)) * rate, currency=ccy),
            transaction_currency=ccy, properties=props))
    try:
        txn_portfolios_api.upsert_transactions(scope=PORTFOLIO_SCOPE, code=port_code, transaction_request=transactions)
        cash_loaded += len(transactions)
        print(f"  {port_code}: {len(transactions)} cash transactions")
    except lu.ApiException as e:
        body = e.body[:300] if e.body else e.reason
        print(f"  {port_code}: ERROR {body}")
print(f"  Total cash: {cash_loaded}")

print(f"\n✅ NB03 Complete: {len(all_codes) + len(benchmarks)} portfolios, {total_loaded + cash_loaded} transactions")